In [1]:
import pandas as pd

In [2]:
#Visualización de los datos (primeras 5 filas)
df = pd.read_csv("Datos_MinisterioHacienda.csv")
df.head()

,id_contrato,estado_contrato,fecha_de_firma,fecha_de_inicio_del_contrato,fecha_de_fin_del_contrato,tipo_de_contrato,modalidad_de_contratacion,duraci_n_del_contrato,dias_adicionados,valor_del_contrato,valor_pagado,valor_pendiente_de_pago,es_pyme,es_grupo,g_nero_representante_legal
0,CO1.PCCNTR.307131,Cerrado,2018-01-28T00:00:00.000,2018-01-18T00:00:00.000,2018-12-31T00:00:00.000,Prestación de servicios,Contratación directa,No definido,0,68757333.0,66999333.0,1758000.0,No,No,No Definido
1,CO1.PCCNTR.4849630,Cerrado,2023-04-11T00:00:00.000,2023-04-11T00:00:00.000,2023-08-09T00:00:00.000,Prestación de servicios,Contratación directa,4 Mes(es),0,10195712.0,10195712.0,0.0,No,No,Mujer
2,CO1.PCCNTR.5952790,Cerrado,2024-02-19T00:00:00.000,2024-02-21T00:00:00.000,2024-10-20T00:00:00.000,Prestación de servicios,Contratación directa,9 Mes(es),0,56700000.0,56490000.0,210000.0,No,No,No Definido
3,CO1.PCCNTR.9045405,En ejecución,2026-01-24T00:00:00.000,2026-01-27T00:00:00.000,2026-09-30T00:00:00.000,Prestación de servicios,Contratación directa,251 Dia(s),0,38228400.0,23361800.0,14866600.0,No,No,Hombre
4,CO1.PCCNTR.2732719,Cerrado,2021-08-18T00:00:00.000,2021-08-18T00:00:00.000,2022-12-13T00:00:00.000,Prestación de servicios,Contratación directa,331 Dia(s),0,117038423.0,116111909.0,926513.0,No,No,No Definido


In [3]:
df.info() 
#Revisamos los tipos de datos y cantidad de valores por columna.

<class 'pandas.DataFrame'>
RangeIndex: 4592 entries, 0 to 4591
Data columns (total 15 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   id_contrato                   4592 non-null   str    
 1   estado_contrato               4592 non-null   str    
 2   fecha_de_firma                4410 non-null   str    
 3   fecha_de_inicio_del_contrato  4401 non-null   str    
 4   fecha_de_fin_del_contrato     4544 non-null   str    
 5   tipo_de_contrato              4592 non-null   str    
 6   modalidad_de_contratacion     4592 non-null   str    
 7   duraci_n_del_contrato         4588 non-null   str    
 8   dias_adicionados              4592 non-null   int64  
 9   valor_del_contrato            4592 non-null   float64
 10  valor_pagado                  4592 non-null   float64
 11  valor_pendiente_de_pago       4592 non-null   float64
 12  es_pyme                       4592 non-null   str    
 13  es_grupo      

In [4]:
# reviso valores atipicos, ceros y nulos antes de limpiar
print(df.isna().sum())
print((df['valor_del_contrato'] == 0).sum())
print(df['valor_del_contrato'].describe())

id_contrato                       0
estado_contrato                   0
fecha_de_firma                  182
fecha_de_inicio_del_contrato    191
fecha_de_fin_del_contrato        48
tipo_de_contrato                  0
modalidad_de_contratacion         0
duraci_n_del_contrato             4
dias_adicionados                  0
valor_del_contrato                0
valor_pagado                      0
valor_pendiente_de_pago           0
es_pyme                           0
es_grupo                          0
g_nero_representante_legal        0
dtype: int64
79
count    4.592000e+03
mean     1.559006e+11
std      1.053309e+13
min      0.000000e+00
25%      3.741322e+07
50%      6.356134e+07
75%      1.046464e+08
max      7.137671e+14
Name: valor_del_contrato, dtype: float64


In [5]:
# quito el contrato con valor imposible (713 billones, error de digitacion)
df = df[df['valor_del_contrato'] < 1e12].copy()
# paso las 3 columnas de fecha de texto a formato fecha
for c in ['fecha_de_firma','fecha_de_inicio_del_contrato','fecha_de_fin_del_contrato']:
    df[c] = pd.to_datetime(df[c], errors='coerce')

# calculo la duracion en dias como  fecha fin menos fecha inicio
df['duracion_dias'] = (df['fecha_de_fin_del_contrato'] - df['fecha_de_inicio_del_contrato']).dt.days 

In [6]:
# Quitar espacios y unificar el formato de texto en estado y modalidad
for c in ["estado_contrato", "modalidad_de_contratacion"]:
    df[c] = df[c].str.strip().str.capitalize()
print(df["estado_contrato"].unique())
print(df["modalidad_de_contratacion"].unique())

<StringArray>
[          'Cerrado',      'En ejecución',          'Aprobado',
        'Modificado',          'Borrador',         'Cancelado',
            'Cedido', 'Enviado proveedor',         'Terminado',
     'En aprobación']
Length: 10, dtype: str
<StringArray>
[                                       'Contratación directa',
                                              'Mínima cuantía',
                         'Selección abreviada subasta inversa',
                          'Contratación directa (con ofertas)',
                                          'Licitación pública',
                        'Selección abreviada de menor cuantía',
                                 'Concurso de méritos abierto',
 'Seleccion abreviada menor cuantia sin manifestacion interes',
                 'Contratación régimen especial (con ofertas)',
                             'Licitación pública obra publica',
                               'Contratación régimen especial']
Length: 11, dtype: str


In [7]:
# Revisar si existen contratos duplicados
duplicados = df["id_contrato"].duplicated().sum()
print("Contratos duplicados:", duplicados)

Contratos duplicados: 0


In [8]:
# Revisar contratos sin fecha de firma
sin_fecha = df[df["fecha_de_firma"].isna()]

print("Contratos sin fecha de firma:", len(sin_fecha))
print("Valor total de contratos sin fecha:", sin_fecha["valor_del_contrato"].sum())

Contratos sin fecha de firma: 181
Valor total de contratos sin fecha: 58408220806.07


In [9]:
# Revisar si los contratos sin fecha de firma tienen fecha de inicio
print("Sin fecha de firma:", df["fecha_de_firma"].isna().sum())

print( "Sin fecha de firma pero con fecha de inicio:",
    (df["fecha_de_firma"].isna() & df["fecha_de_inicio_del_contrato"].notna()).sum()
)

print(
    "Sin fecha de firma ni fecha de inicio:",
    (df["fecha_de_firma"].isna() & df["fecha_de_inicio_del_contrato"].isna()).sum()
)

Sin fecha de firma: 181
Sin fecha de firma pero con fecha de inicio: 10
Sin fecha de firma ni fecha de inicio: 171


In [10]:
sin_anio = df[
    df["fecha_de_firma"].isna() &
    df["fecha_de_inicio_del_contrato"].isna()
]

print("Contratos sin fecha para asignar año:", len(sin_anio))
print("Valor total sin año:", sin_anio["valor_del_contrato"].sum())
print("Porcentaje de contratos:", len(sin_anio) / len(df) * 100)
print("Porcentaje del valor total:",
      sin_anio["valor_del_contrato"].sum() / df["valor_del_contrato"].sum() * 100)

Contratos sin fecha para asignar año: 171
Valor total sin año: 40349906222.42
Porcentaje de contratos: 3.7246787192332826
Porcentaje del valor total: 1.8957246999087705


In [11]:
df["fecha_analisis"] = df["fecha_de_firma"].fillna(
    df["fecha_de_inicio_del_contrato"]
)

# Extraer el año
df["anio"] = df["fecha_analisis"].dt.year.astype("Int64")

print("Contratos sin año:", df["anio"].isna().sum())
print("Años encontrados:", sorted(df["anio"].dropna().unique()))

Contratos sin año: 171
Años encontrados: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]


In [12]:
print("Valores negativos:", (df["valor_del_contrato"] < 0).sum())
print("Valores iguales a cero:", (df["valor_del_contrato"] == 0).sum())

print("\nResumen del valor de los contratos:")
print(df["valor_del_contrato"].describe())

Valores negativos: 0
Valores iguales a cero: 79

Resumen del valor de los contratos:
count    4.591000e+03
mean     4.636177e+08
std      1.340275e+10
min      0.000000e+00
25%      3.741215e+07
50%      6.354877e+07
75%      1.046031e+08
max      8.979613e+11
Name: valor_del_contrato, dtype: float64


In [13]:
# Quito espacios y unifico mayúsculas en tipo de contrato
df["tipo_de_contrato"] = (
    df["tipo_de_contrato"]
    .str.strip()
    .str.capitalize()
)

print(df["tipo_de_contrato"].value_counts(dropna=False))

tipo_de_contrato
Prestación de servicios           4210
Compraventa                        159
Otro                                99
Suministros                         36
Seguros                             23
Consultoría                         22
Obra                                 8
Arrendamiento de inmuebles           7
Decreto 092 de 2017                  7
Servicios financieros                6
Comodato                             5
Negocio fiduciario                   3
Operaciones de crédito público       3
Interventoría                        2
No especificado                      1
Name: count, dtype: int64


In [14]:
df_analisis = df[
    df["anio"].notna()
].copy()

print("Registros originales:", len(df))
print("Registros para análisis:", len(df_analisis))
print("Registros excluidos por falta de año:", len(df) - len(df_analisis))

Registros originales: 4591
Registros para análisis: 4420
Registros excluidos por falta de año: 171


In [15]:
# Estandarizo es_pyme, es_grupo y género (espacios, mayúsculas)
for c in ["es_pyme", "es_grupo"]:
    df_analisis[c] = df_analisis[c].str.strip().str.capitalize()

df_analisis["g_nero_representante_legal"] = (
    df_analisis["g_nero_representante_legal"].str.strip().str.title()
)
df_analisis.loc[
    df_analisis["g_nero_representante_legal"].isin(["No Definido", "No definido"]),
    "g_nero_representante_legal"
] = "No definido"

print(df_analisis["es_pyme"].unique(), df_analisis["es_grupo"].unique())
print(df_analisis["g_nero_representante_legal"].unique())

<StringArray>
['No', 'Si']
Length: 2, dtype: str <StringArray>
['No', 'Si']
Length: 2, dtype: str
<StringArray>
['No definido', 'Mujer', 'Hombre']
Length: 3, dtype: str


In [16]:
# Marco contratos con valor_del_contrato == 0 (no permiten calcular % de ejecución)
df_analisis["valor_contrato_cero"] = df_analisis["valor_del_contrato"] == 0
print("Contratos con valor_del_contrato = 0:", df_analisis["valor_contrato_cero"].sum())
# Se mantienen en el df pero se excluirán del cálculo de % de ejecución

Contratos con valor_del_contrato = 0: 17
